In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
import pytorch_lightning as pl

class CryptoDataset(Dataset):
    def __init__(self, features: pd.DataFrame, labels: pd.DataFrame):
        """
        Args:
            features (pd.DataFrame): DataFrame containing the preprocessed feature data.
            labels (pd.DataFrame): DataFrame containing the unprocessed labels.
        """
        self.features = features.values  # Convert features to NumPy array
        self.labels = labels['&-class']  # Extract the labels column

        # Encode the labels (e.g., 'up'/'down') into integers
        self.label_encoder = LabelEncoder()
        self.labels_encoded = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.features)

    def __getitem__(self, idx):
        """
        Returns a single sample from the dataset, indexed by `idx`.
        """
        feature = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels_encoded[idx], dtype=torch.long)
        return feature, label

class CryptoDataModule(pl.LightningDataModule):
    def __init__(self, directory_path, batch_size=64, train_split=0.8):
        super().__init__()
        self.directory_path = directory_path
        self.batch_size = batch_size
        self.train_split = train_split

    def setup(self, stage=None):
        """
        Load data and split into training and validation datasets.
        """
        features_path = os.path.join(self.directory_path, 'features_filtered.parquet')
        labels_path = os.path.join(self.directory_path, 'labels_filtered.parquet')

        print("Loading parquet files...")
        df_features = pd.read_parquet(features_path)
        df_labels = pd.read_parquet(labels_path)
        print("Parquet files loaded successfully.")

        # Create dataset
        dataset = CryptoDataset(features=df_features, labels=df_labels)

        # Split dataset into training and validation sets
        train_size = int(self.train_split * len(dataset))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

# Main execution
if __name__ == "__main__":
    # Directory path
    directory_path = '/allah/data/parquet'

    # Initialize DataModule
    data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

    # Setup datasets
    data_module.setup()

    # Test train DataLoader
    train_loader = data_module.train_dataloader()
    for batch_features, batch_labels in train_loader:
        print(f"Features batch shape: {batch_features.shape}")
        print(f"Labels batch shape: {batch_labels.shape}")
        break

    # Test validation DataLoader
    val_loader = data_module.val_dataloader()
    for batch_features, batch_labels in val_loader:
        print(f"Validation Features batch shape: {batch_features.shape}")
        print(f"Validation Labels batch shape: {batch_labels.shape}")
        break


In [ ]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F

class CryptoPricePredictor(pl.LightningModule):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super(CryptoPricePredictor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # First hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization for better training stability
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout for regularization

            nn.Linear(hidden_dim, hidden_dim),  # Second hidden layer
            nn.BatchNorm1d(hidden_dim),         # Batch normalization
            nn.ReLU(),                          # Activation function
            nn.Dropout(dropout_rate),           # Dropout

            nn.Linear(hidden_dim, output_dim)   # Output layer
        )

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Training Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for binary classification
        preds = outputs.argmax(dim=1)
        tp = ((preds == 1) & (labels == 1)).float().sum()  # True Positives
        fp = ((preds == 1) & (labels == 0)).float().sum()  # False Positives
        precision = tp / (tp + fp + 1e-8)  # Adding epsilon to avoid division by zero
        self.log('train_precision', precision, on_step=True, on_epoch=True, prog_bar=True)
        
        return loss

    def validation_step(self, batch, batch_idx):
        features, labels = batch
        outputs = self(features)
        loss = F.cross_entropy(outputs, labels)
        
        # Validation Accuracy
        acc = (outputs.argmax(dim=1) == labels).float().mean()
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log('val_acc', acc, on_step=True, on_epoch=True, prog_bar=True)
        
        # Precision for binary classification
        preds = outputs.argmax(dim=1)
        tp = ((preds == 1) & (labels == 1)).float().sum()  # True Positives
        fp = ((preds == 1) & (labels == 0)).float().sum()  # False Positives
        precision = tp / (tp + fp + 1e-8)  # Adding epsilon to avoid division by zero
        self.log('val_precision', precision, on_step=True, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)


In [16]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

# Define input dimensions based on the features
input_dim = 86  # Replace with the actual number of features in your dataset

# Initialize the model
model = CryptoPricePredictor(input_dim=input_dim, hidden_dim=64, output_dim=2)

# Initialize the data module
directory_path = '/allah/data/parquet'  # Update with your actual directory path
data_module = CryptoDataModule(directory_path=directory_path, batch_size=64)

# Initialize EarlyStopping callback
early_stopping = EarlyStopping(
    monitor="val_loss",  # Metric to monitor
    mode="min",          # "min" for minimizing (e.g., loss), "max" for maximizing (e.g., accuracy)
    patience=7,          # Number of epochs with no improvement before stopping
    verbose=True         # Print messages when stopping
)

# Initialize the Trainer with the EarlyStopping callback
trainer = Trainer(max_epochs=1000, callbacks=[early_stopping])

# Train the model
trainer.fit(model, datamodule=data_module)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name | Type   | Params | Mode 
----------------------------------------
0 | fc1  | Linear | 5.6 K  | train
1 | fc2  | Linear | 130    | train
----------------------------------------
5.7 K     Trainable params
0         Non-trainable params
5.7 K     Total params
0.023     Total estimated model params size (MB)
2         Modules in train mode
0         Modules in eval mode


Loading parquet files...
Parquet files loaded successfully.
                                                                            

/allah/freqtrade/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/allah/freqtrade/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Epoch 0:   4%|▍         | 33/810 [00:00<00:04, 194.24it/s, v_num=3, train_loss_step=0.694, train_acc_step=0.516, train_precision_step=0.640]

Epoch 0: 100%|██████████| 810/810 [00:05<00:00, 156.18it/s, v_num=3, train_loss_step=0.689, train_acc_step=0.516, train_precision_step=0.485, val_loss_step=0.701, val_acc_step=0.469, val_precision_step=0.500, val_loss_epoch=0.689, val_acc_epoch=0.538, val_precision_epoch=0.545, train_loss_epoch=0.692, train_acc_epoch=0.523, train_precision_epoch=0.526]

...

Epoch 64: 100%|██████████| 810/810 [00:05<00:00, 156.54it/s, v_num=3, train_loss_step=0.639, train_acc_step=0.594, train_precision_step=0.550, val_loss_step=0.832, val_acc_step=0.438, val_precision_step=0.471, val_loss_epoch=0.633, val_acc_epoch=0.628, val_precision_epoch=0.616, train_loss_epoch=0.611, train_acc_epoch=0.654, train_precision_epoch=0.657]

Metric val_loss improved by 0.001 >= min_delta = 0.0. New best score: 0.633


Epoch 71: 100%|██████████| 810/810 [00:05<00:00, 154.49it/s, v_num=3, train_loss_step=0.632, train_acc_step=0.641, train_precision_step=0.688, val_loss_step=0.807, val_acc_step=0.344, val_precision_step=0.333, val_loss_epoch=0.636, val_acc_epoch=0.629, val_precision_epoch=0.644, train_loss_epoch=0.608, train_acc_epoch=0.656, train_precision_epoch=0.659]

Monitored metric val_loss did not improve in the last 7 records. Best score: 0.633. Signaling Trainer to stop.


Epoch 71: 100%|██████████| 810/810 [00:05<00:00, 154.35it/s, v_num=3, train_loss_step=0.632, train_acc_step=0.641, train_precision_step=0.688, val_loss_step=0.807, val_acc_step=0.344, val_precision_step=0.333, val_loss_epoch=0.636, val_acc_epoch=0.629, val_precision_epoch=0.644, train_loss_epoch=0.608, train_acc_epoch=0.656, train_precision_epoch=0.659]
